In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# Load Dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
try:
    df = pd.read_csv(url)
except Exception:
    df = pd.read_csv("Telco-Customer-Churn.csv")

# Data preprocessing
if 'customerID' in df.columns:
    df.drop(columns=['customerID'], inplace=True)

# Convert TotalCharges to numeric and update blank values to median
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(" ", np.nan), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

X_raw = df.drop(columns=['Churn']).copy()
y_raw = df['Churn'].copy()

# ==============================================================
# ONE-HOT ENCODING FOR CATEGORICAL FEATURES
# ==============================================================
# pd.get_dummies converts all categorical/object columns to dummy 0/1 binary features
# drop_first=True prevents collinearity (dummy variable trap) for Logistic Regression
X = pd.get_dummies(X_raw, drop_first=True)

# Encode target variable explicitly: 'No' -> 0, 'Yes' -> 1
y = y_raw.map({'No': 0, 'Yes': 1}).astype(int)

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save Scaler for test/Streamlit inference
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")

# Export Test Dataset (with one-hot encoded columns)
test_df = pd.DataFrame(X_test, columns=X.columns)
test_df['target'] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Saved 'test_data.csv' successfully.")

# Model Initialization
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(n_estimators=100, max_depth=25, min_samples_split=10, random_state=42)
}

metrics_list = []

# Training & Metric Calculation
for name, model in models.items():
    # Fit model
    if name in ["Logistic Regression", "kNN"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else y_pred
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    # Save model pickle file
    file_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    joblib.dump(model, f"model/{file_name}.pkl")

    # Metrics computation
    metrics_list.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    })

# Output Results Table
results_df = pd.DataFrame(metrics_list)
print("\n=== MODEL PERFORMANCE METRICS ===")
print(results_df.to_string(index=False))

Saved 'test_data.csv' successfully.

=== MODEL PERFORMANCE METRICS ===
           ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.8070 0.8416     0.6584  0.5668 0.6092 0.4843
           Decision Tree    0.7942 0.8284     0.6296  0.5455 0.5845 0.4507
                     kNN    0.7601 0.7835     0.5511  0.5187 0.5344 0.3734
             Naive Bayes    0.6558 0.8096     0.4269  0.8663 0.5719 0.3951
Random Forest (Ensemble)    0.8041 0.8401     0.6713  0.5134 0.5818 0.4639
